# 7.5節：テンソル

バージョン1.10.10で動作確認しています．

事前にインストールが必要なパッケージ
+ LinearAlgebra
+ TensorOperations
+ TensorToolbox 
+ ImageTransformations
+ Images
+ Plots
+ Random（再現性のため）

## 7.5.5節：Juliaによるテンソル分解

In [ ]:
using LinearAlgebra

I, J, K = 3, 4, 2
X = rand(I, J, K) # X は 3 × 4 × 2 のランダムテンソル
A = rand(2, I) # 2 × I の行列
# A ×₁ X （新しいテンソル Y を作る）
Y = zeros(2, J, K)
for j = 1:J, k = 1:K
    Y[:, j, k] = A * X[:, j, k]
end
Y

In [ ]:
using TensorOperations

@tensor Z[ℓ, j, k] := A[ℓ, i] * X[i, j, k]

In [ ]:
norm(Y-Z)

In [ ]:
using TensorToolbox

X=rand(5,4,3)
R=3 # 近似ランクの設定
Kt = cp_als(X, R)
Xhat = full(Kt) 

## 7.5.6節：再び画像の圧縮

In [ ]:
using LinearAlgebra, TensorToolbox, ImageTransformations

In [ ]:
# 書籍内では提供していない find_closest_tt_rank 関数の定義

# -------------------------------------------------------------
# 最大TTランク（構造的上限 + maxdim の制約）の計算
#  dims: 各モードの次元（例: (2,2,2,...,2)）
#  maxdim: 許容する最大ランク
# TT-rank の構造的上限は 「左側の次元積 と 右側の次元積 の最小値」
# -------------------------------------------------------------
function get_tt_ranks(maxdim::Int, dims::Vector{Int})
    N = length(dims)
    N < 2 && return Int[]

    # 左側の累積次元積と右側の累積次元積
    left = cumprod(dims)[1:end-1]
    right = reverse(cumprod(reverse(dims)))[2:end]

    return min.(min.(left, right), maxdim)
end

# -------------------------------------------------------------
# TTランクが tt_ranks のときの総パラメータ数の計算
# Core_k のサイズは r[k] × dims[k] × r[k+1]
# -------------------------------------------------------------
function calculate_tt_params(dims::Vector{Int}, tt_ranks::Vector{Int})
    N = length(dims)
    N ==0 && return 0
    r = [1; tt_ranks; 1]   # 端点は必ず 1
    return sum(r[1:end-1] .* dims .* r[2:end])
end

# -------------------------------------------------------------
# 目標パラメータ数 param_target に最も近い TT-rank を探索
# 構造的最大ランクは概ね 2^(N/2) で十分
# -------------------------------------------------------------
function find_closest_tt_rank(param_target::Int, dims::Vector{Int}; max_r::Int=2^(length(dims)÷2))
    N = length(dims)
    N == 0 && return 0, Int[]
    max_r = max(max_r, 1)
    best_r = 1
    best_ranks = get_tt_ranks(best_r, dims) 
    best_diff = abs(calculate_tt_params(dims, best_ranks) - param_target)
    for r in 2:max_r
        ranks = get_tt_ranks(r, dims)
        diff = abs(calculate_tt_params(dims, ranks) - param_target)
        if diff < best_diff
            best_diff = diff
            best_r = r
            best_ranks = ranks
        end
    end
    return best_r, best_ranks
end



In [ ]:
# -------------------------------------------------------------
# SVD vs CPD vs TTD の比較実験
# imgname: 画像名（指定方法は，下記の実行例参照）
# param: 各手法が使えるパラメータ数の上限（同等比較用）
# -------------------------------------------------------------
function SVDvsCPDvsTTD(img_input::Union{String,AbstractArray}, param::Int)
    # 画像の読み込みと正規化
    img = Float64.(gray.(imresize(img_input, (256,256)))) # 256x256 にリサイズ
    norm_img = norm(img)

    # =========================================================
    # 1. SVD（行列として扱う）
    # =========================================================
    r_svd = Int(floor(param / 2^9))   # 簡単な経験的なランク推定

    F = svd(img) # 特異値分解
    U, S, Vt = F.U, F.S, F.Vt

    img_svd_r = U[:,1:r_svd] * Diagonal(S[1:r_svd]) * Vt[1:r_svd,:] # 低ランク近似
    err_svd = norm(img - img_svd_r) / norm_img
    param_svd = r_svd * (2^9 + 1)

    # =========================================================
    # 2. CPD（テンソル化して CP-ALS）
    # =========================================================
    # テンソル化
    tensor_dims = ntuple(_ -> 2, 16)   # 2×2×...×2 = 256×256
    img_tensor = reshape(img, tensor_dims)

    r_cpd = Int(floor(param / 2^5))   # 簡単な経験的な近似ランク推定

    cpd_factors = cp_als(img_tensor, r_cpd) # CP-ALS による分解

    img_cpd = reshape(full(cpd_factors), 256, 256) # 画像の再構成
    err_cpd = norm(img - img_cpd) / norm_img
    param_cpd = length(cpd_factors.lambda) + sum(length.(cpd_factors.fmat))

    # =========================================================
    # 3. TTD（TT-SVD）
    # =========================================================
    r_ttd, ranks_list = find_closest_tt_rank(param, collect(tensor_dims))  # TTランクの決定

    ttd_factors = TTsvd(img_tensor; reqrank = ranks_list) # TT-SVD による分解

    img_ttd = reshape(full(ttd_factors), 256, 256) # 画像の再構成

    err_ttd = norm(img - img_ttd) / norm_img
    param_ttd = sum(length.(ttd_factors.cores))

    # =========================================================
    # 4. 図の作成
    # =========================================================
    p = plot(
        plot(Gray.(img),      title="Original",         framestyle=:none),
        plot(Gray.(img_svd_r), title="SVD (r=$r_svd)",   framestyle=:none),
        plot(Gray.(img_cpd),   title="CPD (r=$r_cpd)",   framestyle=:none),
        plot(Gray.(img_ttd),   title="TTD (r=$r_ttd)",   framestyle=:none),
        layout=(1,4), size=(1200,300)
    )

    return p, err_svd, err_cpd, err_ttd, param_svd, param_cpd, param_ttd
end

In [ ]:
using Images, Plots
image = load("cat.png")
image = Gray.(image)

using Random
Random.seed!(1234); # using Randomが必要

for param in [2^10, 2^11, 2^12, 2^13, 2^14, 2^15, 2^16]
    p, er_svd, er_cpd, er_ttd, p_svd, p_cpd, p_ttd = SVDvsCPDvsTTD(image, param)
    savefig("SVDvsCPDvsTTD_cat_$(param).pdf")
    println("\nparam = $(param)")
    println("SVD Error: $er_svd, Parameters: $p_svd")
    println("CPD Error: $er_cpd, Parameters: $p_cpd")
    println("TTD Error: $er_ttd, Parameters: $p_ttd")
end